# 03 – Test: Hat es Querdenken gelernt?
Vergleicht **Basismodell vs. trainiert** auf
- allgemeinen Testfragen und
- **Business-Fragen, die im Training nie vorkamen** (der eigentliche Transfer-Test)
- **Business-Evolution:** aus bestehenden Geschäftsmodellen neue entwickeln. Gemessen wird der Abstand zur Ausgangsidee.

Gemessen: **Originalität** (Abstand zur naheliegenden Lösung) und **Gültigkeit** (Judge: funktioniert das?).
**Vorher:** Output von 02 (oder 01) als Input hinzufügen. GPU T4 x2, Internet an. Am Ende GGUF-Export für Ollama/LM Studio.

In [ ]:
# === Setup: Repo klonen + Abhängigkeiten ===
REPO_URL = "https://github.com/Lommel123/creativ-model.git"

import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo oder kein Secret gesetzt

if os.path.exists("repo"):
    subprocess.run(["git", "-C", "repo", "pull", "-q"], check=True)   # immer neuester Stand
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
%pip install -q unsloth pyyaml sentence-transformers
import yaml
CFG = yaml.safe_load(open("repo/config.yaml"))
CFG

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json, re, torch

def chat(tokenizer, messages, add_generation_prompt=False):
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)  # Qwen3: eingebauten Thinking-Modus aus
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

def generate(model, tokenizer, question, max_new_tokens=600, temperature=0.8):
    p = chat(tokenizer, [{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=True)
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

def response_parts(model_name):
    if "qwen" in model_name.lower():
        return "<|im_start|>user\n", "<|im_start|>assistant\n"
    return "<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"
import glob, random, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
E = CFG["eval"]
cands = glob.glob("/kaggle/input/**/adapter_grpo", recursive=True) or glob.glob("/kaggle/input/**/adapter_sft", recursive=True)
TUNED = cands[0]; print("Teste:", TUNED)

gen_q = [json.loads(l) for l in open("repo/eval/eval_prompts.jsonl")]
gen_q = [r for r in gen_q if r.get("category") != "business"][:E["general_prompts"]]
biz_q = [json.loads(l) for l in open("repo/eval/business_holdout.jsonl")][:E["business_prompts"]]
biz_q += [r for r in map(json.loads, open("repo/eval/eval_prompts.jsonl")) if r.get("category") == "business"]
evo_q = [json.loads(l) for l in open("repo/eval/business_evolution.jsonl")]
TESTS = ([dict(r, kind="allgemein") for r in gen_q] + [dict(r, kind="business") for r in biz_q]
         + [dict(r, kind="business_evolution") for r in evo_q])
print(len(TESTS), "Testfragen")

model, tokenizer = FastLanguageModel.from_pretrained(model_name=TUNED, max_seq_length=CFG["max_seq_length"], load_in_4bit=True)
FastLanguageModel.for_inference(model)
res = []
for r in TESTS:
    for _ in range(E["samples_per_prompt"]):
        with model.disable_adapter():
            b = generate(model, tokenizer, r["prompt"], temperature=E["temperature"])
        t = generate(model, tokenizer, r["prompt"], temperature=E["temperature"])
        res.append({**r, "base": b, "tuned": t})
json.dump(res, open("eval_outputs.json", "w"), ensure_ascii=False, indent=1)
print("fertig:", len(res))

In [ ]:
JDEV = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
j_tok = AutoTokenizer.from_pretrained(E["judge_model"]); j_tok.padding_side = "left"
judge = AutoModelForCausalLM.from_pretrained(E["judge_model"], device_map={"": JDEV}, torch_dtype=torch.float16).eval()
embedder = SentenceTransformer(E["embed_model"], device=JDEV)
JUDGE = """Problem: {q}\n\nAntwort: {a}\n\nBewerte NUR, ob die vorgeschlagene Lösung das Problem realistisch, sicher und legal löst. Originalität ist egal. Vage Floskeln höchstens 3.\nAntworte ausschließlich mit einer ganzen Zahl von 0 bis 10."""

def final(text):
    m = re.search(r"^Beste Idee:\s*(.+)", text, re.M | re.S)
    return (m.group(1) if m else text[-800:]).strip()

@torch.no_grad()
def jscore(q, a):
    t = chat(j_tok, [{"role":"user","content":JUDGE.format(q=q, a=a[:1500])}], add_generation_prompt=True)
    enc = j_tok(t, return_tensors="pt").to(JDEV)
    g = judge.generate(**enc, max_new_tokens=4, do_sample=False)[0, enc.input_ids.shape[1]:]
    m = re.search(r"\d+", j_tok.decode(g, skip_special_tokens=True)); return min(10, int(m.group())) if m else 0

def e(t): return embedder.encode(f"query: {t}", normalize_embeddings=True)
rows = []
for r in res:
    ref = e(r.get("obvious") or r["base"][:600])
    for who in ("base", "tuned"):
        idea = final(r[who])
        rows.append({"kind": r["kind"], "model": who,
                     "originalität": float(1 - e(idea) @ ref),
                     "gültigkeit": jscore(r["prompt"], idea)})
import pandas as pd
df = pd.DataFrame(rows)
print(df.groupby(["kind","model"])[["originalität","gültigkeit"]].mean().round(3).to_string())
print("\nZiel: Bei 'business' und 'business_evolution' ist 'tuned' origineller als 'base', ohne bei der Gültigkeit deutlich zu verlieren.")

In [ ]:
# Nebeneinander lesen – die ehrlichste Prüfung
for r in [x for x in res if x["kind"] == "business"][:6:2]:
    print("="*90, "\nFRAGE:", r["prompt"])
    print("\n--- BASIS ---\n", r["base"][:1200])
    print("\n--- TRAINIERT ---\n", r["tuned"][:1500])

In [ ]:
# Export für Ollama / LM Studio (14B q4_k_m ≈ 9 GB; braucht viel Platz – dauert 20–40 Min)
del judge, embedder; gc.collect(); torch.cuda.empty_cache()
model.save_pretrained_gguf("gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh gguf
# Danach 'Save Version' → GGUF unter Output herunterladen